<div style="display: flex; justify-content: center; align-items: center; gap: 30px; padding: 25px; background: #f8f9fa; border-radius: 16px; margin-bottom: 30px; border: 1px solid #dee2e6; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
  <div style="text-align: center;">
    <a href="https://www.youtube.com/channel/UCLoDL_MJpkrMizBuuXnRYsg" target="_blank" style="display: inline-flex; align-items: center; justify-content: center; padding: 12px 24px; background-color: #FF0000; color: white; text-decoration: none; border-radius: 10px; font-weight: 700; font-size: 20px; transition: all 0.3s ease; box-shadow: 0 4px 10px rgba(255,0,0,0.3);">
      <img src="https://psv4.userapi.com/s/v1/d2/RUOgl-0SNsJCuTdcv9mtwvjHJX6Q1SeQIJYEMrxvt4aWqRhbPQFVuVuJcLBZaaQF9YC9ThMY7i-Y4iN8PYmUse5eXzC8QNRLQ-mFPwnGjKvduI5GEgHwk3UpIESNqC7CbCh0_bxUHUyQ/Youtube.png" style="height: 12px; margin-right: 12px; filter: drop-shadow(0 2px 4px rgba(0,0,0,0.1));">
    </a>
  </div>
  <div style="text-align: center;">
    <a href="https://m.vkvideo.ru/@yourovskihneiro" target="_blank" style="display: inline-flex; align-items: center; justify-content: center; padding: 12px 24px; background-color: #0077FF; color: white; text-decoration: none; border-radius: 10px; font-weight: 700; font-size: 20px; transition: all 0.3s ease; box-shadow: 0 4px 10px rgba(0,119,255,0.3);">
      <img src="https://psv4.userapi.com/s/v1/d2/grBahfIZGZhQ3W33y9Wubq7_E6v50uWpxiXNHenTFQEq8Gf593NFcXuQORJGRDkQPMMYZj12AbQkS-VEvQWfCGLNPfO99DyiJ27q8dq8RkCk837GTUNUGiYUHbdspU9ZS10eneDjFUaV/Vk-video_1.png" style="height: 12px; margin-right: 12px; filter: drop-shadow(0 2px 4px rgba(0,0,0,0.1));">
    </a>
  </div>
</div>

# 🎤 Qwen3-TTS - Полный набор для синтеза речи

**Возможности:**
- ✅ Синтез речи (TTS) с готовыми и пользовательскими голосами
- ✅ Клонирование голоса (быстрое клонирование за 3 секунды)
- ✅ Дизайн голоса (создание уникальных голосов по текстовому описанию)
- ✅ **Сохранение голосов (Google Drive)**: ваши голоса не исчезнут после перезапуска сессии!

**Требования:** Google Colab с GPU (T4 или выше)

---

## 0. 💾 Подключение Google Drive (для сохранения голосов)

In [10]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive подключен! Теперь ваши голоса будут сохраняться навсегда.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive подключен! Теперь ваши голоса будут сохраняться навсегда.


## 1. 🔧 Настройка окружения

In [11]:
# Проверка доступности GPU
!nvidia-smi

Tue Apr 21 17:47:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             31W /   70W |    9105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
# Установка пакета qwen-tts и зависимостей
!pip install -U qwen-tts gradio soundfile numpy librosa scipy whisper-openai

# Установка FlashAttention 2 для повышения производительности (опционально)
#!pip install -U flash-attn --no-build-isolation

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)


In [13]:
# Проверка установки
import torch
print(f"Версия PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Версия PyTorch: 2.10.0+cu128
CUDA доступна: True
GPU: Tesla T4
Память GPU: 14.6 GB


## 2. 📦 Загрузка моделей

Мы загрузим все три модели:
- **CustomVoice**: Для TTS с предустановленными голосами
- **Base**: Для клонирования голоса
- **VoiceDesign**: Для создания голосов по описанию

In [14]:
#@title 📥 Загрузка моделей { display-mode: "form" }
#@markdown ### Выберите модели для загрузки:
Load_CustomVoice = True #@param {type:"boolean"}
Load_Base = True #@param {type:"boolean"}
Load_VoiceDesign = False #@param {type:"boolean"}

import torch
import gradio as gr
import soundfile as sf
import numpy as np
import tempfile
import os
import json
import shutil
import scipy.io.wavfile as wavfile
import warnings
from pathlib import Path
from qwen_tts import Qwen3TTSModel

# Suppress warnings
warnings.filterwarnings('ignore', message='.*Chunk (non-data) not understood.*')
warnings.filterwarnings('ignore', category=UserWarning)

# Определение типа данных и реализации внимания в зависимости от возможностей GPU
if torch.cuda.is_available():
    device = "cuda:0"
    major, minor = torch.cuda.get_device_capability()
    if major >= 8:
        dtype = torch.bfloat16
        attn_impl = "flash_attention_2"
    else:
        dtype = torch.float16
        attn_impl = "sdpa"
else:
    device = "cpu"
    dtype = torch.float32
    attn_impl = "eager"

print(f"🚀 Устройство: {device}")
print(f"📊 Тип данных: {dtype}")
print(f"⚡ Внимание: {attn_impl}")

# --- Настройка постоянного хранилища на Google Drive ---
GDRIVE_BASE = Path("/content/drive/MyDrive/Qwen3-TTS")
VOICES_DIR = GDRIVE_BASE / "custom_voices"
VOICES_JSON = VOICES_DIR / "voices.json"
DESIGNED_VOICES_DIR = GDRIVE_BASE / "designed_voices"
DESIGNED_VOICES_JSON = DESIGNED_VOICES_DIR / "voices.json"

try:
    VOICES_DIR.mkdir(parents=True, exist_ok=True)
    DESIGNED_VOICES_DIR.mkdir(parents=True, exist_ok=True)
    for p, default_val in [(VOICES_JSON, {}), (DESIGNED_VOICES_JSON, {})]:
        if not p.exists():
            with open(p, "w", encoding="utf-8") as f:
                json.dump(default_val, f, ensure_ascii=False, indent=2)
    print(f"✅ Google Drive подключен: {GDRIVE_BASE}")
except Exception as e:
    print(f"⚠️ Внимание: Не удалось использовать Google Drive ({e}). Использование временной папки.")
    VOICES_DIR = Path("custom_voices")
    VOICES_JSON = VOICES_DIR / "voices.json"
    DESIGNED_VOICES_DIR = Path("designed_voices")
    DESIGNED_VOICES_JSON = DESIGNED_VOICES_DIR / "voices.json"
    VOICES_DIR.mkdir(parents=True, exist_ok=True)
    DESIGNED_VOICES_DIR.mkdir(parents=True, exist_ok=True)
    for p, default_val in [(VOICES_JSON, {}), (DESIGNED_VOICES_JSON, {})]:
        if not p.exists():
            with open(p, "w", encoding="utf-8") as f:
                json.dump(default_val, f, ensure_ascii=False, indent=2)

🚀 Устройство: cuda:0
📊 Тип данных: torch.float16
⚡ Внимание: sdpa
✅ Google Drive подключен: /content/drive/MyDrive/Qwen3-TTS


In [15]:
# Функция загрузки модели с fallback
def load_model(path):
    try:
        return Qwen3TTSModel.from_pretrained(
            path,
            device_map=device,
            dtype=dtype,
            attn_implementation=attn_impl,
        )
    except Exception as e:
        if attn_impl != "eager":
            print(f"⚠️ Ошибка при загрузке {path} с {attn_impl}. Пробую 'eager' fallback...")
            return Qwen3TTSModel.from_pretrained(
                path,
                device_map=device,
                dtype=dtype,
                attn_implementation="eager",
            )
        raise e

# Загрузка выбранных моделей
custom_voice_model = None
clone_model = None
voice_design_model = None

if Load_CustomVoice:
    print("📥 Загрузка CustomVoice...")
    custom_voice_model = load_model("Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice")
    print("✅ CustomVoice загружена!")

if Load_Base:
    print("📥 Загрузка Base (для клонирования)...")
    clone_model = load_model("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
    print("✅ Base загружена!")

if Load_VoiceDesign:
    print("📥 Загрузка VoiceDesign...")
    voice_design_model = load_model("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
    print("✅ VoiceDesign загружена!")

print("\n🎉 Готово! Загруженные модели:")
if custom_voice_model: print("  ✓ CustomVoice (Синтез речи)")
if clone_model: print("  ✓ Base (Клонирование)")
if voice_design_model: print("  ✓ VoiceDesign (Дизайн голоса)")


📥 Загрузка CustomVoice...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ CustomVoice загружена!
📥 Загрузка Base (для клонирования)...
⚠️ Ошибка при загрузке Qwen/Qwen3-TTS-12Hz-1.7B-Base с sdpa. Пробую 'eager' fallback...


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.42 GiB is allocated by PyTorch, and 21.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 3. 🎛️ Запуск веб-интерфейса Gradio

Интерфейс включает:
- **🎙️ Синтез речи**: Использование готовых голосов
- **🎭 Клонирование голоса**: Клонирование по образцу с возможностью сохранения
- **🎨 Дизайн голоса**: Создание голосов по описанию

In [ ]:
VOICE_DATABASE = {
    "🎈 Мультипликация": {
        "Яркий герой": {
            "en": "Voice style: bright cartoon hero, Tone: energetic, optimistic, youthful, Pace: medium-fast, Emotion: confidence, curiosity, Pitch: slightly high, Clarity: very clear, expressive",
            "ru": "Яркий мультяшный герой: энергичный, уверенный, живой."
        },
        "Весёлый помощник": {
            "en": "Voice style: cartoon sidekick, Tone: playful, friendly, Pace: fast, Emotion: excitement, humor, Pitch: high, Expression: exaggerated",
            "ru": "Весёлый помощник, комедийный персонаж."
        },
        "Мультяшный злодей (Юмор)": {
            "en": "Voice style: cartoon villain, Tone: sneaky, theatrical, Pace: medium, Emotion: arrogance, irony, Pitch: mid-low, Expression: dramatic",
            "ru": "Мультяшный злодей без жести — с юмором."
        },
        "Тёмный злодей": {
            "en": "Voice style: dark cartoon villain, Tone: cold, dominant, Pace: slow, Emotion: menace, Pitch: low, Texture: slightly rough",
            "ru": "Тёмный мультяшный антагонист."
        },
        "Рассказчик (Мульт)": {
            "en": "Voice style: cartoon narrator, Tone: warm, storyteller, Pace: medium, Emotion: wonder, Pitch: mid, Clarity: high",
            "ru": "Рассказчик для мультфильмов."
        }
    },
    "🎬 Кино": {
        "Драматичный герой": {
            "en": "Voice style: cinematic hero, Tone: serious, inspiring, Pace: medium-slow, Emotion: determination, Pitch: mid-low, Presence: strong",
            "ru": "Киногерой, драматичный и уверенный."
        },
        "Антигерой": {
            "en": "Voice style: cinematic antihero, Tone: tired, cynical, Pace: slow, Emotion: restrained anger, Pitch: low, Texture: dry",
            "ru": "Антигерой с усталостью в голосе."
        },
        "Реалистичный злодей": {
            "en": "Voice style: realistic movie villain, Tone: calm, threatening, Pace: slow, Emotion: control, Pitch: low, Expression: minimal",
            "ru": "Реалистичный кинозлодей."
        },
        "Командир": {
            "en": "Voice style: military commander, Tone: authoritative, Pace: medium, Emotion: confidence, Pitch: low, Clarity: sharp",
            "ru": "Командир, лидер, генерал."
        },
        "Учёный": {
            "en": "Voice style: movie scientist, Tone: intellectual, calm, Pace: medium-fast, Emotion: curiosity, Pitch: mid, Clarity: precise",
            "ru": "Учёный, аналитик."
        },
        "Детектив": {
            "en": "Voice style: movie detective, Tone: dry, observant, Pace: slow, Emotion: suspicion, Pitch: mid-low, Texture: slightly gravelly",
            "ru": "Кинодетектив, сдержанный."
        },
        "Криминальный босс": {
            "en": "Voice style: crime boss, Tone: calm dominance, Pace: slow, Emotion: control, Pitch: low, Presence: heavy",
            "ru": "Криминальный авторитет."
        },
        "Монолог (Драма)": {
            "en": "Voice style: dramatic monologue, Tone: emotional, intense, Pace: slow, Emotion: inner conflict, Pitch: mid-low, Dynamics: rising and falling",
            "ru": "Драматический монолог."
        },
        "Мудрый наставник": {
            "en": "Voice style: wise mentor, Tone: warm, confident, Pace: medium-slow, Emotion: guidance, Pitch: mid, Clarity: soft but strong",
            "ru": "Наставник, учитель."
        },
        "Предатель": {
            "en": "Voice style: secretive traitor, Tone: quiet, tense, Pace: slow, Emotion: guilt, fear, Pitch: mid-low, Expression: restrained",
            "ru": "Предатель, скрывающий эмоции."
        }
    },
    "🎮 Игра": {
        "Дружелюбный NPC": {
            "en": "Voice style: game npc friendly, Tone: welcoming, Pace: medium, Emotion: kindness, Pitch: mid",
            "ru": "Дружелюбный NPC."
        },
        "Торговец": {
            "en": "Voice style: game merchant, Tone: sly, talkative, Pace: medium-fast, Emotion: interest, Pitch: mid",
            "ru": "Торговец, купец."
        },
        "Финальный босс": {
            "en": "Voice style: game boss villain, Tone: dominant, Pace: slow, Emotion: menace, Pitch: very low",
            "ru": "Финальный босс."
        },
        "Система ИИ": {
            "en": "Voice style: game ai system, Tone: neutral, synthetic, Pace: steady, Emotion: none, Pitch: mid",
            "ru": "Игровой ИИ / интерфейс."
        },
        "Квестодатель": {
            "en": "Voice style: quest giver, Tone: informative, Pace: medium, Emotion: seriousness, Pitch: mid",
            "ru": "Квестодатель."
        },
        "Молодой герой": {
            "en": "Voice style: young game hero, Tone: confident, energetic, Pace: medium-fast, Emotion: determination, Pitch: mid-high, Clarity: clear",
            "ru": "Молодой герой видеоигры, решительный и бодрый."
        },
        "Героиня": {
            "en": "Voice style: female game hero, Tone: strong, calm, Pace: medium, Emotion: confidence, Pitch: mid, Presence: solid",
            "ru": "Героиня игры, уверенная, без излишней агрессии."
        },
        "Мрачный герой": {
            "en": "Voice style: dark game hero, Tone: cold, restrained, Pace: slow, Emotion: inner struggle, Pitch: low-mid",
            "ru": "Мрачный герой с внутренним конфликтом."
        },
        "Маг": {
            "en": "Voice style: magic user, Tone: mysterious, Pace: medium-slow, Emotion: focus, Pitch: mid, Texture: soft",
            "ru": "Маг, заклинатель, мистик."
        },
        "Рыцарь": {
            "en": "Voice style: noble knight, Tone: honorable, firm, Pace: medium, Emotion: duty, Pitch: mid-low",
            "ru": "Рыцарь, благородный защитник."
        },
        "Ассасин": {
            "en": "Voice style: stealth assassin, Tone: quiet, sharp, Pace: slow, Emotion: control, Pitch: low, Volume: subdued",
            "ru": "Ассасин, скрытный и холодный."
        },
        "Безумный злодей": {
            "en": "Voice style: mad game villain, Tone: unstable, Pace: irregular, Emotion: insanity, Pitch: mid-high, Dynamics: chaotic",
            "ru": "Безумный злодей."
        },
        "Старый маг": {
            "en": "Voice style: old wizard npc, Tone: slow, wise, Pace: very slow, Emotion: knowledge, Pitch: low, Texture: breathy",
            "ru": "Старый мудрый маг."
        },
        "Детский NPC": {
            "en": "Voice style: child npc, Tone: innocent, curious, Pace: medium-fast, Emotion: excitement, Pitch: high",
            "ru": "Детский NPC."
        },
        "Королева-злодейка": {
            "en": "Voice style: villain queen, Tone: cold elegance, Pace: slow, Emotion: superiority, Pitch: mid-low",
            "ru": "Злодейка-королева, холодная и властная."
        }
    },
    "🇷🇺 Русские типажи": {
        "Рассказчик": {
            "en": "Voice style: Russian storyteller, Tone: calm, deep, thoughtful, Pace: medium-slow, Emotion: wisdom, nostalgia, Pitch: low-mid, Clarity: clear, steady",
            "ru": "Русский рассказчик, спокойный, вдумчивый, немного философский."
        },
        "Офицер": {
            "en": "Voice style: Russian military officer, Tone: strict, authoritative, Pace: medium, Emotion: discipline, Pitch: low, Clarity: sharp, commanding",
            "ru": "Военный офицер, жёсткий, уверенный, командный."
        },
        "Реалистичный злодей": {
            "en": "Voice style: realistic Russian villain, Tone: cold, restrained, Pace: slow, Emotion: controlled threat, Pitch: low, Expression: minimal",
            "ru": "Реалистичный злодей без карикатуры."
        },
        "Бандит": {
            "en": "Voice style: Russian bandit, Tone: rough, street-wise, Pace: medium, Emotion: arrogance, Pitch: low, Texture: slightly raspy",
            "ru": "Уличный, грубый голос, криминальный типаж."
        },
        "Философ": {
            "en": "Voice style: Russian philosopher, Tone: slow, reflective, Pace: slow, Emotion: contemplation, Pitch: mid-low, Clarity: soft but clear",
            "ru": "Философ, размышляющий, спокойный."
        },
        "Учёный": {
            "en": "Voice style: Russian scientist, Tone: intellectual, focused, Pace: medium-fast, Emotion: curiosity, Pitch: mid, Clarity: precise",
            "ru": "Учёный, аналитик, рациональный."
        },
        "Диктор ТВ": {
            "en": "Voice style: Russian TV announcer, Tone: formal, confident, Pace: medium, Emotion: neutrality, Pitch: mid, Clarity: broadcast-level",
            "ru": "Телеведущий / диктор."
        },
        "Дедушка": {
            "en": "Voice style: elderly Russian man, Tone: tired but wise, Pace: slow, Emotion: experience, Pitch: low, Texture: slightly shaky",
            "ru": "Пожилой мужчина с жизненным опытом."
        },
        "Мужик из деревни": {
            "en": "Voice style: Russian village man, Tone: simple, grounded, Pace: medium-slow, Emotion: sincerity, Pitch: low-mid, Texture: natural",
            "ru": "Деревенский мужик, простой и искренний."
        },
        "Интеллигент": {
            "en": "Voice style: Russian intellectual, Tone: polite, cultured, Pace: medium, Emotion: restraint, Pitch: mid, Clarity: very clean",
            "ru": "Интеллигент, образованный, спокойный."
        }
    },
    "🎌 Аниме": {
        "Герой Сёнен": {
            "en": "Voice style: anime shounen hero, Tone: energetic, loud, Pace: fast, Emotion: passion, Pitch: high, Expression: exaggerated",
            "ru": "Энергичный аниме-герой."
        },
        "Спокойный герой": {
            "en": "Voice style: calm anime hero, Tone: cool, controlled, Pace: slow, Emotion: confidence, Pitch: mid-low",
            "ru": "Спокойный аниме-протагонист."
        },
        "Холодный злодей": {
            "en": "Voice style: cold anime villain, Tone: emotionless, Pace: slow, Emotion: dominance, Pitch: low",
            "ru": "Холодный аниме-злодей."
        },
        "Безумная девушка": {
            "en": "Voice style: anime crazy girl, Tone: playful insanity, Pace: fast, Emotion: obsession, Pitch: high, Dynamics: extreme",
            "ru": "Безумная, эмоциональная девушка."
        },
        "Нежная девушка": {
            "en": "Voice style: soft anime girl, Tone: gentle, shy, Pace: slow, Emotion: tenderness, Pitch: high-soft",
            "ru": "Мягкий, нежный женский голос."
        },
        "Цундере": {
            "en": "Voice style: tsundere archetype, Tone: sharp but emotional, Pace: medium-fast, Emotion: irritation hiding care, Pitch: mid-high",
            "ru": "Цундере — резкая, но чувствительная."
        },
        "Аниме-наставник": {
            "en": "Voice style: anime mentor, Tone: relaxed wisdom, Pace: slow, Emotion: guidance, Pitch: mid-low",
            "ru": "Аниме-наставник."
        },
        "Монолог злодея": {
            "en": "Voice style: villain monologue, Tone: theatrical, Pace: slow, Emotion: arrogance, Pitch: mid-low, Dynamics: dramatic",
            "ru": "Монолог злодея."
        },
        "Маленький герой": {
            "en": "Voice style: anime child hero, Tone: bright, brave, Pace: fast, Emotion: courage, Pitch: high",
            "ru": "Детский герой."
        },
        "Финальный босс": {
            "en": "Voice style: anime final boss, Tone: overwhelming, Pace: slow, Emotion: absolute confidence, Pitch: very low",
            "ru": "Финальный аниме-босс."
        }
    },
    "👻 Хоррор": {
        "Шёпот ужаса": {
            "en": "Voice style: horror whisper, Tone: eerie, Pace: very slow, Emotion: dread, Pitch: low, Volume: quiet",
            "ru": "Шёпот ужаса."
        },
        "Монстр": {
            "en": "Voice style: horror monster, Tone: distorted, Pace: irregular, Emotion: aggression, Pitch: very low",
            "ru": "Монстр."
        },
        "Рассказчик хоррора": {
            "en": "Voice style: horror narrator, Tone: dark, calm, Pace: slow, Emotion: tension, Pitch: low",
            "ru": "Рассказчик хоррора."
        },
        "Жуткий ребёнок": {
            "en": "Voice style: creepy child, Tone: innocent but unsettling, Pace: slow, Pitch: high",
            "ru": "Жуткий детский голос."
        },
        "Голос по рации": {
            "en": "Voice style: distorted radio voice, Tone: cold, Pace: slow, Emotion: isolation, Pitch: mid-low",
            "ru": "Голос по рации."
        }
    },
    "🌑 Тёмные / Психологические": {
        "Тревожный рассказчик": {
            "en": "Voice style: psychological horror narrator, Tone: calm but unsettling, Pace: slow, Emotion: hidden dread, Pitch: low, Clarity: soft, intimate",
            "ru": "Спокойный, но тревожный рассказчик."
        },
        "Внутренний голос": {
            "en": "Voice style: inner voice, Tone: quiet, intimate, Pace: slow, Emotion: doubt, Pitch: mid-low, Volume: low",
            "ru": "Внутренний голос персонажа."
        },
        "Безумный шёпот": {
            "en": "Voice style: mad whisper, Tone: unstable, Pace: irregular, Emotion: paranoia, Pitch: low, Volume: very quiet",
            "ru": "Безумный шёпот."
        },
        "Травмированный герой": {
            "en": "Voice style: traumatized character, Tone: shaky, restrained, Pace: slow, Emotion: fear, suppression, Pitch: mid, Texture: unstable",
            "ru": "Травмированный персонаж."
        },
        "Манипулятор": {
            "en": "Voice style: manipulative voice, Tone: smooth, calm, Pace: slow, Emotion: control, Pitch: mid-low, Expression: subtle",
            "ru": "Манипулятор, говорит мягко и опасно."
        },
        "Сонный паралич": {
            "en": "Voice style: sleep paralysis entity, Tone: cold, inhuman, Pace: very slow, Emotion: oppression, Pitch: very low",
            "ru": "Существо из сонного паралича."
        },
        "Потерянная душа": {
            "en": "Voice style: lost soul, Tone: distant, hollow, Pace: slow, Emotion: emptiness, Pitch: mid-low, Texture: airy",
            "ru": "Потерянная душа."
        },
        "Обрывки памяти": {
            "en": "Voice style: broken memory, Tone: fragmented, Pace: uneven, Emotion: confusion, Pitch: shifting",
            "ru": "Обрывки воспоминаний."
        },
        "Лидер культа": {
            "en": "Voice style: cult leader, Tone: calm authority, Pace: slow, Emotion: devotion, Pitch: low, Presence: hypnotic",
            "ru": "Лидер культа."
        },
        "Признание под пыткой": {
            "en": "Voice style: tortured confession, Tone: weak, emotional, Pace: slow, Emotion: guilt, Pitch: mid, Texture: breathy",
            "ru": "Признание под давлением."
        },
        "Жуткий ребёнок (Темнота)": {
            "en": "Voice style: dark child, Tone: innocent but wrong, Pace: slow, Emotion: eerie calm, Pitch: high-soft",
            "ru": "Жуткий детский голос."
        },
        "Теневая сущность": {
            "en": "Voice style: shadow entity, Tone: barely human, Pace: slow, Emotion: menace, Pitch: very low, Clarity: muffled",
            "ru": "Теневая сущность."
        },
        "Последнее предупреждение": {
            "en": "Voice style: final warning, Tone: urgent but controlled, Pace: medium-slow, Emotion: desperation, Pitch: mid-low",
            "ru": "Последнее предупреждение."
        },
        "Наблюдатель": {
            "en": "Voice style: observer, Tone: detached, Pace: slow, Emotion: neutrality, Pitch: mid-low",
            "ru": "Наблюдатель, без эмоций."
        },
        "Космический ужас": {
            "en": "Voice style: cosmic horror voice, Tone: vast, indifferent, Pace: very slow, Emotion: insignificance, Pitch: extremely low",
            "ru": "Космический ужас, нечеловеческий масштаб."
        }
    },
    "🌌 Концептуальные": {
        "Абстрактный": {
            "en": "Voice style: abstract voice, Tone: non-specific, fluid, Pace: irregular, Emotion: ambiguous, Pitch: shifting, Clarity: unstable",
            "ru": "Абстрактный голос без чёткого характера."
        },
        "Голос сна": {
            "en": "Voice style: dream-like voice, Tone: soft, distant, Pace: slow, Emotion: calm detachment, Pitch: mid-high, Texture: airy",
            "ru": "Голос сна, отстранённый и мягкий."
        },
        "Ночные мысли": {
            "en": "Voice style: late night thoughts, Tone: quiet, personal, Pace: slow, Emotion: reflection, Pitch: mid-low, Volume: low",
            "ru": "Мысли перед сном."
        },
        "Память": {
            "en": "Voice style: memory narrator, Tone: nostalgic, Pace: medium-slow, Emotion: longing, Pitch: mid, Clarity: soft",
            "ru": "Воспоминания, рассказанные голосом."
        },
        "Голос за спиной": {
            "en": "Voice style: voice from behind, Tone: close, unsettling, Pace: slow, Emotion: intrusion, Pitch: low, Volume: intimate",
            "ru": "Голос будто за спиной."
        },
        "Эхо руин": {
            "en": "Voice style: ruined broadcast, Tone: tired, distant, Pace: slow, Emotion: isolation, Pitch: mid-low, Clarity: degraded",
            "ru": "Сообщение из разрушенного мира."
        },
        "Петля времени": {
            "en": "Voice style: time loop voice, Tone: repetitive, weary, Pace: steady, Emotion: resignation, Pitch: mid, Dynamics: cyclical",
            "ru": "Голос, застрявший во времени."
        },
        "Посланник": {
            "en": "Voice style: divine messenger, Tone: calm authority, Pace: slow, Emotion: certainty, Pitch: mid-low, Presence: elevated",
            "ru": "Посланник высшей силы."
        },
        "Падший ангел": {
            "en": "Voice style: fallen angel, Tone: beautiful sorrow, Pace: slow, Emotion: regret, Pitch: mid-high, Texture: soft but heavy",
            "ru": "Падший ангел, печальный и величественный."
        },
        "Постчеловек": {
            "en": "Voice style: post-human voice, Tone: detached, Pace: slow, Emotion: neutrality, Pitch: mid, Texture: unnatural",
            "ru": "Постчеловеческий голос."
        },
        "Голос пустоты": {
            "en": "Voice style: voice of the void, Tone: empty, vast, Pace: very slow, Emotion: nothingness, Pitch: extremely low",
            "ru": "Голос пустоты."
        },
        "Мета-рассказчик": {
            "en": "Voice style: meta narrator, Tone: self-aware, Pace: medium, Emotion: irony, Pitch: mid, Clarity: very clear",
            "ru": "Самоосознающий рассказчик."
        },
        "Голос автора": {
            "en": "Voice style: author voice, Tone: confident, personal, Pace: medium-slow, Emotion: intention, Pitch: mid",
            "ru": "Голос автора."
        },
        "Тишина": {
            "en": "Voice style: restrained minimalism, Tone: extremely calm, Pace: very slow, Emotion: stillness, Pitch: neutral, Dynamics: minimal",
            "ru": "Минималистичный, почти безэмоциональный стиль."
        }
    },
    "😂 Мем": {
        "Переигрыш": {
            "en": "Voice style: meta meme, Tone: hyperbolic, Pace: fast, Emotion: overreaction, Pitch: high",
            "ru": "Мемный переигрыш."
        },
        "Сухой голос": {
            "en": "Voice style: deadpan meme, Tone: flat, Pace: slow, Emotion: irony, Pitch: mid",
            "ru": "Сухой мем."
        },
        "Мемный рассказчик": {
            "en": "Voice style: meme narrator, Tone: sarcastic, Pace: medium, Emotion: humor, Pitch: mid",
            "ru": "Мемный рассказчик."
        },
        "Ироничный ИИ": {
            "en": "Voice style: ironic ai voice, Tone: robotic but funny, Pace: medium, Pitch: mid",
            "ru": "Ироничный ИИ."
        },
        "Хаос": {
            "en": "Voice style: chaotic meme, Tone: unstable, Pace: very fast, Emotion: madness, Pitch: random",
            "ru": "Полный мемный хаос."
        }
    },
    "📖 Аудиокнига": {
        "Классика": {
            "en": "Voice style: classic audiobook narrator, Tone: warm, Pace: medium-slow, Emotion: calm, Pitch: mid",
            "ru": "Классическая аудиокнига."
        },
        "Драма": {
            "en": "Voice style: dramatic narration, Tone: emotional, Pace: slow, Pitch: mid-low",
            "ru": "Драматичный рассказ."
        },
        "Сказочник": {
            "en": "Voice style: fairytale narrator, Tone: magical, Pace: medium, Emotion: wonder, Pitch: mid",
            "ru": "Сказочник."
        },
        "Документалка": {
            "en": "Voice style: documentary narrator, Tone: neutral, Pace: steady, Pitch: mid",
            "ru": "Документальный стиль."
        },
        "Мрачный": {
            "en": "Voice style: dark narration, Tone: gloomy, Pace: slow, Pitch: low",
            "ru": "Мрачный рассказчик."
        }
    },
    "⚙️ Экспериментальные": {
        "Синтетическая": {
            "en": "Voice style: synthetic, Tone: artificial, Pace: steady, Pitch: mid",
            "ru": "Синтетический голос."
        },
        "ИИ будущего": {
            "en": "Voice style: futuristic ai, Tone: clean, Pace: precise, Pitch: mid-high",
            "ru": "ИИ будущего."
        },
        "Сломанный ИИ": {
            "en": "Voice style: broken ai, Tone: unstable, Pace: irregular, Pitch: shifting",
            "ru": "Сломанный ИИ."
        },
        "Устаревший ИИ": {
            "en": "Voice style: obsolete ai, Tone: polite but outdated, Pace: medium, Emotion: fading usefulness, Pitch: mid, Texture: synthetic-soft",
            "ru": "Устаревший ИИ."
        },
        "Робот-помощник": {
            "en": "Voice style: robot assistant, Tone: helpful, Pace: medium, Pitch: mid",
            "ru": "Голос помощника."
        },
        "Терминал": {
            "en": "Voice style: terminal system, Tone: cold, Pace: slow, Pitch: low",
            "ru": "Системный терминал."
        }
    }
}


# --- Вспомогательные функции ---
def _normalize_audio(wav, eps=1e-12, clip=True):
    x = np.asarray(wav)
    if np.issubdtype(x.dtype, np.integer):
        info = np.iinfo(x.dtype)
        y = x.astype(np.float32) / max(abs(info.min), info.max)
    else:
        y = x.astype(np.float32)
    if clip: y = np.clip(y, -1.0, 1.0)
    if y.ndim > 1: y = np.mean(y, axis=-1)
    return y

def _audio_to_tuple(audio):
    if audio is None: return None
    if isinstance(audio, str):
        sr, wav = wavfile.read(audio)
        return _normalize_audio(wav), int(sr)
    if isinstance(audio, tuple):
        sr, wav = audio
        return _normalize_audio(wav), int(sr)
    return None

def get_saved_voices():
    try:
        with open(VOICES_JSON, "r", encoding="utf-8") as f: voices = json.load(f)
        return sorted(list(voices.keys()))
    except: return []

def save_voice(name, audio, ref_text, xvector_only):
    if not name or not audio: return "Ошибка: Имя и аудио обязательны.", gr.update(choices=get_saved_voices())
    try:
        audio_tuple = _audio_to_tuple(audio)
        sr, wav = audio_tuple[1], audio_tuple[0]
        audio_path = VOICES_DIR / f"{name}.wav"
        wavfile.write(audio_path, sr, (wav * 32767).astype(np.int16))
        with open(VOICES_JSON, "r", encoding="utf-8") as f: voices = json.load(f)
        voices[name] = {"ref_audio_path": str(audio_path), "ref_text": ref_text, "xvector_only": xvector_only}
        with open(VOICES_JSON, "w", encoding="utf-8") as f: json.dump(voices, f, ensure_ascii=False, indent=2)
        return f"✅ Голос '{name}' сохранен!", gr.update(choices=get_saved_voices(), value=name)
    except Exception as e: return f"❌ Ошибка: {str(e)}", gr.update(choices=get_saved_voices())

def load_voice_data(name):
    if not name: return None, "", False
    try:
        with open(VOICES_JSON, "r", encoding="utf-8") as f: voices = json.load(f)
        v = voices.get(name, {})
        return v.get("ref_audio_path"), v.get("ref_text", ""), v.get("xvector_only", False)
    except: return None, "", False

def delete_voice(name):
    if not name: return "Ошибка: Выберите голос.", gr.update(choices=get_saved_voices())
    try:
        with open(VOICES_JSON, "r", encoding="utf-8") as f: voices = json.load(f)
        if name in voices:
            p = Path(voices[name]["ref_audio_path"])
            if p.exists(): p.unlink()
            del voices[name]
            with open(VOICES_JSON, "w", encoding="utf-8") as f: json.dump(voices, f, ensure_ascii=False, indent=2)
        return f"✅ Голос '{name}' удален.", gr.update(choices=get_saved_voices(), value=None)
    except Exception as e: return f"❌ Ошибка: {str(e)}", gr.update(choices=get_saved_voices())

# --- Основные функции генерации ---
SPEAKERS = {
    "Vivian": "Яркий, дерзкий молодой женский голос",
    "Serena": "Теплый, нежный женский голос",
    "Uncle_Fu": "Зрелый мужской голос с низким тембром",
    "Dylan": "Молодежный мужской голос (Пекинский диалект)",
    "Eric": "Живой мужской голос (Сычуаньский диалект)",
    "Ryan": "Динамичный мужской голос с сильным ритмом (English)",
    "Aiden": "Солнечный американский мужской голос",
    "Ono_Anna": "Игривый японский женский голос",
    "Sohee": "Теплый корейский женский голос с эмоциями"
}

LANGUAGES = ["Auto", "Russian", "English", "Chinese", "Japanese", "Korean", "German", "French", "Portuguese", "Spanish", "Italian"]

def generate_tts(text, speaker, language, instruct):
    if custom_voice_model is None:
        return None, "❌ Модель CustomVoice не загружена! Вернитесь к ячейке загрузки и включите Load_CustomVoice."
    if not text.strip(): return None, "Введите текст."
    try:
        wavs, sr = custom_voice_model.generate_custom_voice(text=text, language=None if language=="Auto" else language, speaker=speaker, instruct=instruct if instruct.strip() else None)
        out = tempfile.mktemp(suffix=".wav"); sf.write(out, wavs[0], sr)
        return out, f"✅ Готово! Диктор: {speaker}"
    except Exception as e: return None, f"❌ Ошибка: {str(e)}"

def clone_voice_fn(text, ref_audio, ref_text, language, xvector_only):
    if clone_model is None:
        return None, "❌ Модель Base не загружена! Вернитесь к ячейке загрузки и включите Load_Base."
    if not text.strip() or ref_audio is None: return None, "Введите текст и загрузите аудио."
    try:
        ref_input = _audio_to_tuple(ref_audio)
        wavs, sr = clone_model.generate_voice_clone(text=text, language=None if language=="Auto" else language, ref_audio=ref_input, ref_text=ref_text if ref_text.strip() else None, x_vector_only_mode=xvector_only)
        out = tempfile.mktemp(suffix=".wav"); sf.write(out, wavs[0], sr)
        return out, "✅ Голос успешно клонирован!"
    except Exception as e: return None, f"❌ Ошибка: {str(e)}"

# --- Voice Design Saving Functions ---
def get_saved_designs():
    try:
        with open(DESIGNED_VOICES_JSON, "r", encoding="utf-8") as f: designs = json.load(f)
        return sorted(list(designs.keys()))
    except: return []

def save_design(name, description, seed):
    if not name or not description: return "Ошибка: Имя и описание обязательны.", gr.update(choices=get_saved_designs())
    try:
        with open(DESIGNED_VOICES_JSON, "r", encoding="utf-8") as f: designs = json.load(f)
        designs[name] = {"description": description, "seed": seed}
        with open(DESIGNED_VOICES_JSON, "w", encoding="utf-8") as f: json.dump(designs, f, ensure_ascii=False, indent=2)
        return f"✅ Дизайн '{name}' сохранен!", gr.update(choices=get_saved_designs(), value=name)
    except Exception as e: return f"❌ Ошибка: {str(e)}", gr.update(choices=get_saved_designs())

def load_design_data(name):
    if not name: return "", 42
    try:
        with open(DESIGNED_VOICES_JSON, "r", encoding="utf-8") as f: designs = json.load(f)
        d = designs.get(name, {})
        return d.get("description", ""), d.get("seed", 42)
    except: return "", 42

def delete_design(name):
    if not name: return "Ошибка: Выберите дизайн.", gr.update(choices=get_saved_designs())
    try:
        with open(DESIGNED_VOICES_JSON, "r", encoding="utf-8") as f: designs = json.load(f)
        if name in designs:
            del designs[name]
            with open(DESIGNED_VOICES_JSON, "w", encoding="utf-8") as f: json.dump(designs, f, ensure_ascii=False, indent=2)
        return f"✅ Дизайн '{name}' удален.", gr.update(choices=get_saved_designs(), value=None)
    except Exception as e: return f"❌ Ошибка: {str(e)}", gr.update(choices=get_saved_designs())

def design_voice_fn(text, voice_description, language):
    if voice_design_model is None:
        return None, "❌ Модель VoiceDesign не загружена! Вернитесь к ячейке загрузки и включите Load_VoiceDesign."
    if not text.strip() or not voice_description.strip(): return None, "Введите текст и описание."
    try:
        wavs, sr = voice_design_model.generate_voice_design(text=text, language=None if language=="Auto" else language, instruct=voice_description)
        out = tempfile.mktemp(suffix=".wav"); sf.write(out, wavs[0], sr)
        return out, "✅ Голос успешно создан!"
    except Exception as e: return None, f"❌ Ошибка: {str(e)}"

# --- Интерфейс ---
css = """
    .gradio-container {
        max-width: 1200px !important;
        margin: auto !important;
        padding: 0 1rem !important;
    }
    .header-container {
        text-align: center;
        padding: 2.5rem 1rem;
        background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 50%, #9333ea 100%);
        border-radius: 20px;
        margin-bottom: 2rem;
        box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1);
        color: white !important;
    }
    .header-container h1 {
        color: white !important;
        font-size: 2.5rem !important;
        font-weight: 800 !important;
        margin-bottom: 0.5rem !important;
    }
    .header-container p {
        color: rgba(255,255,255,0.95) !important;
        font-size: 1.1rem !important;
    }
    .feature-badge {
        display: inline-flex;
        background: rgba(255,255,255,0.2);
        padding: 0.35rem 0.9rem;
        border-radius: 20px;
        margin: 0.3rem;
        font-size: 0.85rem;
        color: white;
        border: 1px solid rgba(255,255,255,0.1);
    }
    footer { display: none !important; }
    .custom-footer {
        text-align: center;
        margin-top: 2rem;
        padding: 1.5rem;
        border-top: 1px solid #eee;
        color: #666;
    }
    .custom-footer a {
        color: #4f46e5;
        text-decoration: none;
        font-weight: 600;
    }
    @media (max-width: 768px) {
        .header-container h1 { font-size: 1.8rem !important; }
        .gradio-container { padding: 0 0.5rem !important; }
    }
"""
with gr.Blocks(title="Qwen3-TTS Suite", theme=gr.themes.Soft(), css=css) as demo:
    gr.HTML('''<div class="header-container"><h1>🎙️ Qwen3-TTS</h1><p>Высококачественный синтез речи с функциями клонирования и дизайна голоса</p><div style="margin-top: 1rem;"><span class="feature-badge">🎨 Дизайн голоса</span><span class="feature-badge">🎭 Клонирование</span><span class="feature-badge">🗣️ Свои голоса</span></div></div>''')

    with gr.Tabs():
        with gr.TabItem("🎙️ Синтез речи"):
            with gr.Row():
                with gr.Column():
                    tts_text = gr.Textbox(label="Текст для озвучки", placeholder="Введите текст...", lines=4)
                    tts_speaker = gr.Dropdown(choices=list(SPEAKERS.keys()), value="Ryan", label="Диктор")
                    tts_speaker_info = gr.Markdown(f"*{SPEAKERS['Ryan']}*")
                    tts_language = gr.Dropdown(choices=LANGUAGES, value="Auto", label="Язык")
                    tts_instruct = gr.Textbox(label="Инструкция по стилю (Опционально)", placeholder="Напр: 'Говори радостно'", lines=2)
                    tts_btn = gr.Button("🎵 Озвучить", variant="primary")
                with gr.Column():
                    tts_output = gr.Audio(label="Результат", type="filepath")
                    tts_status = gr.Markdown()
            tts_speaker.change(fn=lambda s: f"*{SPEAKERS.get(s, '')}*", inputs=tts_speaker, outputs=tts_speaker_info)
            tts_btn.click(fn=generate_tts, inputs=[tts_text, tts_speaker, tts_language, tts_instruct], outputs=[tts_output, tts_status])

        with gr.TabItem("🎭 Клонирование"):
            with gr.Row():
                with gr.Column():
                    with gr.Accordion("💾 Сохраненные голоса", open=True):
                        with gr.Row():
                            saved_dropdown = gr.Dropdown(label="Выбрать сохраненный", choices=get_saved_voices(), scale=3)
                            refresh_btn = gr.Button("🔄", scale=1)
                        save_name = gr.Textbox(label="Имя для сохранения", placeholder="Введите имя...")
                        with gr.Row():
                            btn_save = gr.Button("💾 Сохранить", variant="secondary"); btn_del = gr.Button("🗑️ Удалить", variant="stop")
                        save_status = gr.Markdown()
                    clone_ref_audio = gr.Audio(label="Образец голоса (3+ сек)", type="filepath")
                    clone_ref_text = gr.Textbox(label="Текст образца (Опционально)", lines=2)
                    clone_xvector = gr.Checkbox(label="Только x-vector (текст не нужен, качество ниже)", value=False)
                    clone_text = gr.Textbox(label="Целевой текст", lines=4)
                    clone_language = gr.Dropdown(choices=LANGUAGES, value="Auto", label="Язык")
                    clone_btn = gr.Button("🎭 Клонировать", variant="primary")
                with gr.Column():
                    clone_output = gr.Audio(label="Результат", type="filepath")
                    clone_status = gr.Markdown()

            refresh_btn.click(fn=lambda: gr.update(choices=get_saved_voices()), inputs=[], outputs=[saved_dropdown])
            btn_save.click(save_voice, inputs=[save_name, clone_ref_audio, clone_ref_text, clone_xvector], outputs=[save_status, saved_dropdown])
            btn_del.click(delete_voice, inputs=[saved_dropdown], outputs=[save_status, saved_dropdown])
            saved_dropdown.change(load_voice_data, inputs=[saved_dropdown], outputs=[clone_ref_audio, clone_ref_text, clone_xvector])
            clone_btn.click(fn=clone_voice_fn, inputs=[clone_text, clone_ref_audio, clone_ref_text, clone_language, clone_xvector], outputs=[clone_output, clone_status])

        with gr.TabItem("🎨 Дизайн голоса"):
            with gr.Row():
                with gr.Column():
                    with gr.Accordion("💾 Сохраненные дизайны", open=True):
                        with gr.Row():
                            design_saved_dropdown = gr.Dropdown(label="Выбрать сохраненный", choices=get_saved_designs(), scale=3)
                            design_refresh_btn = gr.Button("🔄", scale=1)
                        design_save_name = gr.Textbox(label="Имя для сохранения", placeholder="Введите имя...")
                        with gr.Row():
                            design_btn_save = gr.Button("💾 Сохранить", variant="secondary")
                            design_btn_del = gr.Button("🗑️ Удалить", variant="stop")
                        design_save_status = gr.Markdown()
                    design_text = gr.Textbox(label="Текст для озвучки", lines=4)
                    design_desc = gr.Textbox(label="Описание голоса", placeholder="Напр: 'Пожилой мужчина с хриплым голосом'", lines=4)
                    design_seed = gr.Number(label="Seed (для воспроизводимости)", value=42, precision=0)
                    design_language = gr.Dropdown(choices=LANGUAGES, value="Auto", label="Язык")
                    design_btn = gr.Button("🎨 Создать голос", variant="primary")
                with gr.Column():
                    design_output = gr.Audio(label="Результат", type="filepath")
                    design_status = gr.Markdown()

            # Design tab events
            design_refresh_btn.click(fn=lambda: gr.update(choices=get_saved_designs()), inputs=[], outputs=[design_saved_dropdown])
            design_btn_save.click(save_design, inputs=[design_save_name, design_desc, design_seed], outputs=[design_save_status, design_saved_dropdown])
            design_btn_del.click(delete_design, inputs=[design_saved_dropdown], outputs=[design_save_status, design_saved_dropdown])
            design_saved_dropdown.change(load_design_data, inputs=[design_saved_dropdown], outputs=[design_desc, design_seed])
            design_btn.click(fn=design_voice_fn, inputs=[design_text, design_desc, design_language], outputs=[design_output, design_status])
        with gr.TabItem("📚 База дизайнов голосов"):
            gr.Markdown("### 🔍 Исследуйте библиотеку готовых стилей голосов")

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("#### 📁 Категории")
                    with gr.Row(variant="compact"):
                        db_cat_btns = []
                        for category in VOICE_DATABASE.keys():
                            db_cat_btns.append(gr.Button(category, size="sm", variant="secondary"))

                    gr.Markdown("#### 🎙️ Голоса")
                    db_voice_list = gr.Dropdown(
                        label="Выберите голос из категории",
                        choices=[],
                        interactive=True
                    )

                with gr.Column(scale=1):
                    gr.Markdown("#### 📄 Детали голоса")
                    db_voice_en = gr.Code(label="EN Instruct (для модели)", language="markdown", interactive=False)
                    db_voice_ru = gr.Textbox(label="Описание на русском", interactive=False)

                    db_apply_btn = gr.Button("🎨 Использовать в дизайне", variant="primary")
                    db_apply_status = gr.Markdown()

            # Database Events
            def select_db_category(cat):
                voices = list(VOICE_DATABASE.get(cat, {}).keys())
                return gr.update(choices=voices, value=voices[0] if voices else None)

            for btn in db_cat_btns:
                btn.click(select_db_category, inputs=[btn], outputs=[db_voice_list])

            def load_db_voice_details(cat, voice):
                if not cat or not voice: return "", ""
                details = VOICE_DATABASE.get(cat, {}).get(voice, {})
                return details.get("en", ""), details.get("ru", "")

            current_db_cat = gr.State("")
            for btn in db_cat_btns:
                btn.click(lambda c: c, inputs=[btn], outputs=[current_db_cat])

            db_voice_list.change(
                load_db_voice_details,
                inputs=[current_db_cat, db_voice_list],
                outputs=[db_voice_en, db_voice_ru]
            )

            def apply_db_to_design(instruct):
                return instruct, "✅ Инструкция скопирована во вкладку 'Дизайн голоса'!"

            db_apply_btn.click(
                apply_db_to_design,
                inputs=[db_voice_en],
                outputs=[design_desc, db_apply_status]
            )


    gr.HTML('''<div class="custom-footer">Powered by <a href="https://github.com/QwenLM/Qwen3-TTS" target="_blank">Qwen3-TTS</a> | Модифицированно <a href="https://www.youtube.com/channel/UCLoDL_MJpkrMizBuuXnRYsg" target="_blank">Максимом Юровских</a></div>''')

    # Refresh dropdown on page load
    demo.load(fn=lambda: gr.update(choices=get_saved_voices()), inputs=None, outputs=[saved_dropdown])

demo.launch(share=True, debug=True)


## 4. 📝 Быстрые примеры (без UI)

In [ ]:
from IPython.display import Audio, display

text = "Привет! Добро пожаловать в Qwen3-TTS. Это демонстрация синтеза речи высокого качества."
wavs, sr = custom_voice_model.generate_custom_voice(
    text=text,
    language="Russian",
    speaker="Ryan",
    instruct="Говори с энтузиазмом"
)

print("🎵 Результат TTS:")
display(Audio(wavs[0], rate=sr))

### Поддерживаемые языки
Русский, Английский, Китайский, Японский, Корейский, Немецкий, Французский, Португальский, Испанский, Итальянский

### Требования к памяти
- Модели 1.7B: ~8-10 ГБ видеопамяти
- Модели 0.6B: ~4-6 ГБ видеопамяти (используйте их при ограниченных ресурсах)